# LifeMate baseline v7 trên Kaggle
Planner hướng dẫn một bước → caller chọn tool + arguments + values → summarizer quyết định kết thúc hoặc quay lại planner. Planner không trả done. Lỗi env được đưa qua summarizer để xét tiếp tục; guard/budget vẫn chặn lặp. Ba vai trò dùng chung Qwen checkpoint, chưa fine-tune.

Bật GPU/Internet, attach dữ liệu và toàn bộ baseline_v7 (giữ src/, prompts/ và requirements/). MODE=workflow dùng gold text; baseline dùng ASR + agent; all chạy các mode.


### Batch inference
Để lấy số liệu nhanh: chọn `MODE="workflow"`, `INFERENCE_WORKERS=4`, `VLLM_MAX_NUM_SEQS=4`. Các mẫu dùng ToolEnv riêng; vLLM gom các node sẵn sàng. Giảm workers xuống 2 nếu thiếu VRAM.
Batch tăng throughput, không bảo đảm giảm latency từng mẫu. Đo giới hạn 30 giây bằng workers=1 trên A30. Batch có thể đổi output dù seed giống nhau; so sánh cùng cấu hình. Summary có `experiment_wall_s` và `samples_per_second`.


In [ ]:
from pathlib import Path

MODE = "workflow"   # "asr", "workflow", "baseline", or "all"
LIMIT = 50     # 0 = full train split
SAMPLE_ID = None  # Example: "train_1703"; runs only this case in workflow mode
SEED = 42
DATA_ROOT = None  # Example: "/kaggle/input/lifemate/train+public_test"
OUTPUT_DIR = "/kaggle/working/lifemate_baseline_v7"
RUN_MODE = "workflow" if SAMPLE_ID else MODE
RUN_OUTPUT_DIR = str(Path(OUTPUT_DIR) / "single" / SAMPLE_ID) if SAMPLE_ID else OUTPUT_DIR
ASR_MODEL = "vinai/PhoWhisper-medium"
LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LLM_BACKEND = "vllm"  # Change to "transformers" for the old backend
LLM_TEMPERATURE = 0.7
ASR_BATCH_SIZE = 1
# 1: isolated latency; 4: batch ready node requests from four independent samples.
INFERENCE_WORKERS = 1  # Try 4 to collect results faster; reduce to 2 if VRAM is tight
VLLM_MAX_NUM_SEQS = INFERENCE_WORKERS
BATCH_WAIT_MS = 10
SAVE_NODE_INPUTS = False  # True: store exact prompts per node for later training data
STRUCTURED_OUTPUTS = True  # guidance backend; schema compilation preflight runs before model load
VLLM_USE_CUDA_GRAPHS = False
VLLM_MAX_MODEL_LEN = 12288
VLLM_GPU_MEMORY_UTILIZATION = 0.82
VLLM_TENSOR_PARALLEL_SIZE = 1  # Set to 2 only when two GPUs are visible
VLLM_ATTENTION_BACKEND = "TRITON_ATTN"  # Avoid FlashInfer JIT linker errors on Kaggle T4
VLLM_DTYPE = "float16"  # Tesla T4 does not support bfloat16
MAX_PROMPT_TOKENS = 11500
MAX_NEW_TOKENS = 512
MAX_NO_PROGRESS = 2
MAX_TOOL_CALLS = 10
MAX_DECISIONS = 24  # Rounds; each has planner, at most one caller, and summarizer

SCRIPT_DIR = None  # Có thể đặt đường dẫn baseline_v7 thủ công
if SCRIPT_DIR is None:
    paths = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        paths += [p.parents[2] for p in kaggle_input.rglob("src/lifemate_v7/pipeline.py")]
    for ancestor in (Path.cwd(), *Path.cwd().parents):
        for candidate in (ancestor, ancestor / "baseline_v7"):
            if (candidate / "src/lifemate_v7/pipeline.py").is_file():
                paths.append(candidate)
    if not paths:
        raise FileNotFoundError("Attach toàn bộ baseline_v7, hoặc đặt SCRIPT_DIR thủ công.")
    SCRIPT_DIR = paths[0]
SCRIPT_DIR = Path(SCRIPT_DIR)
required_files = ("run.py", "src/lifemate_v7/pipeline.py", "src/lifemate_v7/contracts.py",
    "src/lifemate_v7/runner.py", "src/lifemate_v7/models.py", "src/lifemate_v7/common.py",
    "src/lifemate_v7/prompts/planner.txt", "src/lifemate_v7/prompts/caller.txt",
    "src/lifemate_v7/prompts/summarizer.txt",
    "requirements/vllm.txt", "requirements/transformers.txt")
if SAMPLE_ID:
    required_files += ("run_one.py",)
for name in required_files:
    if not (SCRIPT_DIR / name).is_file():
        raise FileNotFoundError(f"Thiếu {name}; cần attach đầy đủ cấu trúc baseline_v7.")
RUN_SCRIPT = SCRIPT_DIR / ("run_one.py" if SAMPLE_ID else "run.py")
print("Script:", RUN_SCRIPT)
print("Output:", RUN_OUTPUT_DIR)


In [ ]:
import subprocess
import sys

requirements = "vllm.txt" if LLM_BACKEND == "vllm" and RUN_MODE != "asr" else "transformers.txt"
requirement_path = Path(SCRIPT_DIR) / "requirements" / requirements
if not requirement_path.is_file():
    raise FileNotFoundError(requirement_path)
install_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "on", "-r", str(requirement_path)]
print("Installing:", requirement_path, flush=True)
subprocess.run(install_command, check=True, timeout=1800)
print("Dependency installation completed.")


In [ ]:
import os
import subprocess
import sys

command = [sys.executable, str(RUN_SCRIPT)]
if SAMPLE_ID:
    command.append(SAMPLE_ID)
else:
    command.extend(["--mode", MODE, "--limit", str(LIMIT)])
command.extend([
    "--seed", str(SEED),
    "--output-dir", RUN_OUTPUT_DIR,
    "--asr-model", ASR_MODEL,
    "--llm-model", LLM_MODEL,
    "--llm-backend", LLM_BACKEND,
    "--llm-temperature", str(LLM_TEMPERATURE),
    "--asr-batch-size", str(ASR_BATCH_SIZE),
    "--inference-workers", str(1 if SAMPLE_ID else INFERENCE_WORKERS),
    "--vllm-max-num-seqs", str(VLLM_MAX_NUM_SEQS),
    "--batch-wait-ms", str(BATCH_WAIT_MS),
    "--vllm-max-model-len", str(VLLM_MAX_MODEL_LEN),
    "--vllm-gpu-memory-utilization", str(VLLM_GPU_MEMORY_UTILIZATION),
    "--vllm-tensor-parallel-size", str(VLLM_TENSOR_PARALLEL_SIZE),
    "--vllm-attention-backend", VLLM_ATTENTION_BACKEND,
    "--vllm-dtype", VLLM_DTYPE,
    "--max-prompt-tokens", str(MAX_PROMPT_TOKENS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-no-progress", str(MAX_NO_PROGRESS),
    "--max-tool-calls", str(MAX_TOOL_CALLS),
    "--max-decisions", str(MAX_DECISIONS),
])
if SAVE_NODE_INPUTS:
    command.append("--save-node-inputs")
if not STRUCTURED_OUTPUTS:
    command.append("--no-structured-outputs")
if VLLM_USE_CUDA_GRAPHS:
    command.append("--vllm-use-cuda-graphs")
if DATA_ROOT:
    command.extend(["--data-root", DATA_ROOT])
run_env = os.environ.copy()
run_env["PYTHONHASHSEED"] = str(SEED)
run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
if LLM_BACKEND == "vllm" and RUN_MODE != "asr":
    run_env["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
print("Running:", " ".join(command))
subprocess.run(command, check=True, env=run_env)


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

out = Path(RUN_OUTPUT_DIR)
for name in ("asr", "workflow_gold", "baseline_asr"):
    summary = out / name / "summary.json"
    chart = out / name / "charts.png"
    if summary.exists():
        print("\n", name)
        print(json.dumps(json.loads(summary.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    if chart.exists():
        display(Image(filename=str(chart)))
print("Download result folder:", out)
